<a href="https://colab.research.google.com/github/claudiodanielpc/medium/blob/main/rezago_python_svy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import httpx
import polars as pl
from io import BytesIO
from zipfile import ZipFile
!pip install "svy[report]"
import svy

In [56]:
# -------------------------------------------------
# URL ENIGH 2024 – Viviendas
# -------------------------------------------------
url_vivi = (
    "https://www.inegi.org.mx/contenidos/programas/"
    "enigh/nc/2024/microdatos/enigh2024_ns_viviendas_csv.zip"
)

print("Descargando...")
response = httpx.get(url_vivi, follow_redirects=True, timeout=120.0)
response.raise_for_status()

# -------------------------------------------------
# Leer ZIP en memoria
# -------------------------------------------------
with ZipFile(BytesIO(response.content)) as zf:
    print("Archivos en ZIP:", zf.namelist())
    csv_name = [f for f in zf.namelist() if f.endswith(".csv")][0]

    with zf.open(csv_name) as f:
        viviendas = pl.read_csv(
            f,
            encoding="latin1",
            null_values=["&", "N/A", ""]
        )

# -------------------------------------------------
# Ajustar variable excusado
# -------------------------------------------------
valor_excusado = (
    3 if viviendas.select(pl.col("excusado").n_unique()).item() == 3 else 2
)

# -------------------------------------------------
# BLOQUE 1: Formato de variables
# -------------------------------------------------
viviendas = viviendas.with_columns([
    #Crear contador de viviendas
    pl.lit(1).cast(pl.Int64).alias("viviendas"),

    # tot_resid y num_cuarto SIEMPRE numéricos
    pl.col("tot_resid")
      .cast(pl.Float64, strict=False)
      .fill_null(0)
      .alias("tot_resid"),

    pl.col("num_cuarto")
      .cast(pl.Float64, strict=False)
      .fill_null(0)
      .alias("num_cuarto"),

    # mat*
    pl.selectors.matches("^mat")
      .cast(pl.Int64, strict=False)
      .fill_null(0),
])

# -------------------------------------------------
# BLOQUE 2: Clave de entidad y rezago
# -------------------------------------------------
viviendas = viviendas.with_columns([
    # cve_ent (folioviv es int → cast a string)
    pl.when(pl.col("folioviv").cast(pl.Utf8).str.len_chars() == 9)
      .then(pl.lit("0") + pl.col("folioviv").cast(pl.Utf8).str.slice(0, 1))
    .when(pl.col("folioviv").cast(pl.Utf8).str.len_chars() == 10)
      .then(pl.col("folioviv").cast(pl.Utf8).str.slice(0, 2))
    .otherwise(None)
    .alias("cve_ent"),

    # rezago
    pl.when(
        (
            (pl.col("num_cuarto") > 0) &
            ((pl.col("tot_resid") / pl.col("num_cuarto")) > 2.5)
        )
        |
        (pl.col("mat_pared").is_between(1, 6))
        |
        (pl.col("mat_techos").is_in([1, 2, 3, 4, 6, 7, 9]))
        |
        (pl.col("mat_pisos") == 1)
        |
        (pl.col("excusado") == valor_excusado)
    )
    .then(pl.lit("En rezago"))
    .otherwise(pl.lit("Fuera de rezago"))
    .alias("rezago")
])


# -------------------------------------------------
# BLOQUE 3: nombre de entidad
# -------------------------------------------------
viviendas = viviendas.with_columns(
    pl.col("cve_ent").replace({
        "01": "Aguascalientes",
        "02": "Baja California",
        "03": "Baja California Sur",
        "04": "Campeche",
        "05": "Chiapas",
        "06": "Chihuahua",
        "07": "Coahuila",
        "08": "Colima",
        "09": "Ciudad de México",
        "10": "Durango",
        "11": "Guanajuato",
        "12": "Guerrero",
        "13": "Hidalgo",
        "14": "Jalisco",
        "15": "México",
        "16": "Michoacán",
        "17": "Morelos",
        "18": "Nayarit",
        "19": "Nuevo León",
        "20": "Oaxaca",
        "21": "Puebla",
        "22": "Querétaro Arteaga",
        "23": "Quintana Roo",
        "24": "San Luis Potosí",
        "25": "Sinaloa",
        "26": "Sonora",
        "27": "Tabasco",
        "28": "Tamaulipas",
        "29": "Tlaxcala",
        "30": "Veracruz",
        "31": "Yucatán",
        "32": "Zacatecas",
    }).alias("nom_ent")
)
print(viviendas.head())
print(f"Shape: {viviendas.shape}")

Descargando...
Archivos en ZIP: ['viviendas.csv']
shape: (5, 86)
┌───────────┬──────────┬───────────┬────────────┬───┬───────────┬─────────┬────────────┬───────────┐
│ folioviv  ┆ tipo_viv ┆ mat_pared ┆ mat_techos ┆ … ┆ viviendas ┆ cve_ent ┆ rezago     ┆ nom_ent   │
│ ---       ┆ ---      ┆ ---       ┆ ---        ┆   ┆ ---       ┆ ---     ┆ ---        ┆ ---       │
│ i64       ┆ i64      ┆ i64       ┆ i64        ┆   ┆ i64       ┆ str     ┆ str        ┆ str       │
╞═══════════╪══════════╪═══════════╪════════════╪═══╪═══════════╪═════════╪════════════╪═══════════╡
│ 100001901 ┆ 7        ┆ 8         ┆ 10         ┆ … ┆ 1         ┆ 01      ┆ Fuera de   ┆ Aguascali │
│           ┆          ┆           ┆            ┆   ┆           ┆         ┆ rezago     ┆ entes     │
│ 100001902 ┆ 1        ┆ 8         ┆ 10         ┆ … ┆ 1         ┆ 01      ┆ Fuera de   ┆ Aguascali │
│           ┆          ┆           ┆            ┆   ┆           ┆         ┆ rezago     ┆ entes     │
│ 100001904 ┆ 1        ┆ 8

### Definir diseño muestral

In [57]:
# Diseño muestral
design = svy.Design(
    stratum="est_dis",
    psu="upm",
    wgt="factor"
    )
sample = svy.Sample(data=viviendas, design=design)

print(sample)

╭─────────────────────────── Sample ────────────────────────────╮
│ Survey Data:                                                  │
│   Number of rows: 90324                                       │
│   Number of columns: 89                                       │
│   Number of strata: 681                                       │
│   Number of PSUs: 10569                                       │
│                                                               │
│ Survey Design:                                                │
│                                                               │
│    Field               Value                                  │
│    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━                          │
│    Row index           svy_row_index                          │
│    Stratum             est_dis                                │
│    PSU                 upm                                    │
│    SSU                 None                                   │
│    Weight              factor                                 │
│    With replacement    False                                  │
│    Prob                None                                   │
│    Hit                 None                                   │
│    MOS                 None                                   │
│    Population size     None                                   │
│    Replicate weights   None                                   │
│                                                               │
╰───────────────────────────────────────────────────────────────╯

### Cálculos de viviendas y rezago total y por entidad

In [58]:
### Calcular total de viviendas
totviv = sample.estimation.total(y="viviendas")

print(totviv)


╭────────────────────────── Estimate: TOTAL (TAYLOR) ───────────────────────────╮
│                                                                               │
│              est             se               lci               uci   cv (%)  │
│  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│  38,356,042.0000   142,516.3169   38,076,680.9559   38,635,403.0441     0.37  │
│                                                                               │
╰───────────────────────────────────────────────────────────────────────────────╯

In [72]:
##Calcular rezago

rezago_total = sample.estimation.total(y="viviendas", by="rezago")

print(rezago_total)

╭─────────────────────────────────── Estimate: TOTAL (TAYLOR) ────────────────────────────────────╮
│                                                                                                 │
│  rezago                        est             se               lci               uci   cv (%)  │
│  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│  En rezago          8,381,545.0000   106,629.1295    8,172,530.1615    8,590,559.8385     1.27  │
│  Fuera de rezago   29,974,497.0000   144,400.9414   29,691,441.7075   30,257,552.2925     0.48  │
│                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────╯

In [66]:
# Rezago por entidad federativa
rezago_ent = sample.categorical.tabulate(
    rowvar="rezago",
    colvar="nom_ent",
    units=svy.TableUnits.COUNT,
)
print(rezago_ent)

╭─────────────────────────────────────── Table ────────────────────────────────────────╮
│ Type=Two-Way                                                                         │
│ Alpha=0.05                                                                           │
│                                                                                      │
│                                         Estima      Std                              │
│ Row               Col                       te      Err       CV    Lower      Upper │
│ ──────────────────────────────────────────────────────────────────────────────────── │
│ En rezago         Aguascalientes        16755.   2076.1   0.1239   12685.   20824.69 │
│                                           0000      571              3086         14 │
│ En rezago         Baja California       474035   23282.   0.0491   428397   519672.8 │
│                                          .0000     2084             .1236        764 │
│ En rezago         Baja California Sur   58987.   3828.5   0.0649   51482.   66491.83 │
│                                           0000      996              1640         60 │
│ En rezago         Campeche              115596   4870.2   0.0421   106049   125142.6 │
│                                          .0000      173             .3809        191 │
│ En rezago         Chiapas               88072.   7561.6   0.0859   73249.   102894.4 │
│                                           0000      884              5487        513 │
│ En rezago         Chihuahua             54659.   2865.5   0.0524   49041.   60276.01 │
│                                           0000      245              9877         23 │
│ En rezago         Ciudad de México      182009   17552.   0.0964   147603   216414.7 │
│                                          .0000     1499             .2069        931 │
│ En rezago         Coahuila              101051   40909.   0.0405   930322   1090705. │
│                                         4.0000     7866             .4756       5244 │
│ En rezago         Colima                441301   18699.   0.0424   404645   477956.5 │
│                                          .0000     8560             .4688        312 │
│ En rezago         Durango               106894   8827.0   0.0826   89591.   124196.8 │
│                                          .0000      751              1327        673 │
│ En rezago         Guanajuato            221803   14532.   0.0655   193315   250290.3 │
│                                          .0000     8630             .6250        750 │
│ En rezago         Guerrero              484667   19271.   0.0398   446890   522443.1 │
│                                          .0000     5575             .8174        826 │
│ En rezago         Hidalgo               157728   11221.   0.0711   135732   179723.9 │
│                                          .0000     2460             .0695        305 │
│ En rezago         Jalisco               196356   20091.   0.1023   156973   235738.5 │
│                                          .0000     0244             .4951        049 │
│ En rezago         Michoacán             385276   27715.   0.0719   330948   439603.4 │
│                                          .0000     1819             .5916        084 │
│ En rezago         Morelos               114985   6545.3   0.0569   102154   127815.1 │
│                                          .0000      023             .8727        273 │
│ En rezago         México                533829   37542.   0.0703   460237   607420.3 │
│                                          .0000     6822             .6870        130 │
│ En rezago         Nayarit               80646.   5321.2   0.0660   70215.   91076.72 │
│                                           0000      467              2713         87 │
│ En rezago         Nuevo León            106370   11873.   0.1116   83095.   129644.8 │
│                                          .0000     7007              1253        7